## This script filters German BASt traffic counting stations within Kiel's bounding box (2021–2025). Station coordinates are converted from UTM32 to WGS84 and checked against the bounding box. Matching raw traffic data is extracted, enriched with station metadata (name, coordinates, directions), and saved as a combined CSV. A separate summary of station coverage per month is also exported.

*Developed with the assistance of [Claude (Anthropic)](https://claude.ai)*

In [1]:
import os
import polars as pl
from pyproj import Transformer
from collections import defaultdict

BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "BASt", "AggregatedDataForKiel")
YEARS    = [2021, 2022, 2023, 2024, 2025]
MONTHS   = [f"{m:02d}" for m in range(1, 13)]
VARIANTS = ["A", "B"]

def load_bounding_box():
    df = pl.read_csv(os.path.join(BASE_DIR, "BoundingBox", "kiel_bb.csv"))
    return df.row(0, named=True)

def get_kiel_stations(meta_file, bb, transformer):
    stations = {}  # Metadata dictionary
    for row in pl.read_csv(meta_file, infer_schema_length=0).iter_rows(named=True):
        try:
            e = float(str(row["Koordinaten_UTM32_E"]).replace(".", "").replace(",", "."))
            n = float(str(row["Koordinaten_UTM32_N"]).replace(".", "").replace(",", "."))
            lat, lon = transformer.transform(e, n)
            if bb["lat_min"] <= lat <= bb["lat_max"] and bb["lon_min"] <= lon <= bb["lon_max"]:
                nr = int(row["Dauerzaehlstellennummer"])
                stations[nr] = {
                    "name": row["Dauerzaehlstellenname"],
                    "richtung_1": row["Himmelsrichtung_Richtung_1"],
                    "richtung_2": row["Himmelsrichtung_Richtung_2"],
                    "lat": lat,
                    "lon": lon,
                }
        except Exception:
            pass
    return stations

def main():
    bb          = load_bounding_box()
    transformer = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=False)
    station_months = defaultdict(set)
    station_meta   = {}
    filtered_data  = []

    for year in YEARS:
        for variant in VARIANTS:
            folder = os.path.join(BASE_DIR, "BASt", "AggregatedDataForSH", f"Data {year}", f"SH_{year}_{variant}_S")
            for month in MONTHS:
                meta_file = os.path.join(folder, f"Meta_{year}_{month}_{variant}_S.csv")
                roh_file  = os.path.join(folder, f"Roh_{year}_{month}_{variant}_S.csv")
                if not os.path.exists(meta_file) or not os.path.exists(roh_file):
                    continue

                stations = get_kiel_stations(meta_file, bb, transformer)
                for nr, meta in stations.items():
                    station_months[nr].add(f"{year}-{month}")
                    station_meta[nr] = meta

                if stations:
                    filtered_data.append(
                        pl.read_csv(roh_file, infer_schema_length=0)
                        .with_columns(pl.col("Zst").str.strip_chars().cast(pl.Int64, strict=False).alias("Zst_int"))
                        .filter(pl.col("Zst_int").is_in(list(stations.keys())))
                        .drop("Zst_int")
                    )

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    meta_df = pl.DataFrame([
        {
            "station":    nr,
            "name":       station_meta[nr]["name"],
            "lat":        station_meta[nr]["lat"],
            "lon":        station_meta[nr]["lon"],
            "richtung_1": station_meta[nr]["richtung_1"],
            "richtung_2": station_meta[nr]["richtung_2"],
        }
        for nr in station_meta
    ])

    (
        pl.concat(filtered_data, how="diagonal")
        .with_columns(pl.col("Zst").str.strip_chars().cast(pl.Int64, strict=False).alias("station"))
        .join(meta_df, on="station", how="left")
        .drop("station")
        .write_csv(os.path.join(OUTPUT_DIR, "kiel_stations_raw.csv"))
    )

    pl.DataFrame([
        {"station": nr, "months_present": len(m), "months": ", ".join(sorted(m))}
        for nr, m in sorted(station_months.items())
    ]).write_csv(os.path.join(OUTPUT_DIR, "kiel_station_count.csv"))

if __name__ == "__main__":
    main()